# Pyama analysis

Edit the **Config** cell, then run the pipeline cells in order:
`segment` → `timeseries` → `plot-timeseries` → `auc` → `plot-auc` → `fit` → `plot-fit`.

Expects ROI stacks from `crop.ipynb` under `WORKSPACE/roi/`. Experiment settings live in this notebook; the `pyama` package provides core/services only.

## Config

Edit **workspace**, **slide mapping**, and **timing** below. Everything under *Advanced* can usually stay at the defaults.

In [ ]:
from pathlib import Path

# Dataset root (roi/, mask/, timeseries/, results/)
WORKSPACE = Path(r"C:\path\to\dataset")

# Slide mapping: slide_channel -> positions, signal/mask channels, sample name
# signal_channel is always 1, mask_channel is always 0; edit ranges and sample_name
SLIDE_MAPPING = {
    0: {
        "positions": list(range(0, 12)),
        "signal_channel": 1,
        "mask_channel": 0,
        "sample_name": "untreated",
    },
    1: {
        "positions": list(range(12, 24)),
        "signal_channel": 1,
        "mask_channel": 0,
        "sample_name": "vehicle",
    },
    2: {
        "positions": list(range(24, 36)),
        "signal_channel": 1,
        "mask_channel": 0,
        "sample_name": "lipofectamine",
    },
    3: {
        "positions": list(range(36, 48)),
        "signal_channel": 1,
        "mask_channel": 0,
        "sample_name": "electroporation",
    },
    4: {
        "positions": list(range(48, 60)),
        "signal_channel": 1,
        "mask_channel": 0,
        "sample_name": "low_dose",
    },
    5: {
        "positions": list(range(60, 72)),
        "signal_channel": 1,
        "mask_channel": 0,
        "sample_name": "high_dose",
    },
}

# Frame interval (minutes) and fit onset search window
INTERVAL_MINUTES = 10.0
MAX_ONSET_MINUTES = 120.0  # 0 = translation_onset fixed at 0

# --- Advanced (usually leave as-is) ---
JOBS = 4
VARIATION_RADIUS = 2
GAUSSIAN_SIGMA = 1.0
FORCE_MASKS = False
RUN_SEGMENT = True
RUN_TIMESERIES = True
RUN_CHECK_SEGMENT = False  # optional mask QA videos
WRITE_SLIDE_JSON = False  # optional: persist mapping under WORKSPACE/slide.json
PLOT_COLUMNS = 3

## Setup

In [ ]:
from pyama.core import (
    resolve_slide_path,
    slide_channel_labels,
    validate_slide_mapping,
    workspace_timeseries_dir,
    write_slide_mapping,
)
from pyama.services import (
    auc,
    check_segment,
    fit,
    plot_auc,
    plot_fit,
    plot_timeseries,
    segment,
    timeseries,
)

workspace = WORKSPACE.expanduser().resolve()
if not workspace.is_dir():
    raise FileNotFoundError(f"Workspace not found: {workspace}")

mapping = validate_slide_mapping(SLIDE_MAPPING)
labels = slide_channel_labels(mapping)

if WRITE_SLIDE_JSON:
    slide_path = write_slide_mapping(mapping, resolve_slide_path(workspace))
    print(f"Wrote slide mapping: {slide_path}")

print(f"Workspace: {workspace}")
for sc, entry in mapping.items():
    print(
        f"  sc={sc} sample={entry.sample_name!r} "
        f"signal={entry.signal_channel} mask={entry.mask_channel} "
        f"positions={entry.positions}"
    )

## Segment

In [ ]:
if RUN_SEGMENT:
    result = segment.run_segment(
        workspace=workspace,
        mapping=mapping,
        variation_radius=VARIATION_RADIUS,
        gaussian_sigma=GAUSSIAN_SIGMA,
        force=FORCE_MASKS,
        jobs=JOBS,
        on_mask_written=lambda sc, out_dir, n: print(
            segment.format_written_masks_message(sc, out_dir, n)
        ),
    )
    if result.skipped_positions:
        print(segment.format_skipped_positions_message(result.skipped_positions))
else:
    print("Skipped segment")

## Check segment (optional)

In [ ]:
if RUN_CHECK_SEGMENT:
    result = check_segment.run_check_segment(
        workspace=workspace,
        mapping=mapping,
        jobs=JOBS,
        on_video_written=lambda video: print(
            check_segment.format_written_check_segment_video_message(video)
        ),
    )
    if result.skipped_positions:
        print(check_segment.format_skipped_positions_message(result.skipped_positions))
else:
    print("Skipped check_segment")

## Timeseries

In [ ]:
if RUN_TIMESERIES:
    result = timeseries.run_timeseries(
        workspace=workspace,
        mapping=mapping,
        jobs=JOBS,
        on_csv_written=lambda sc, csv_path, n: print(
            timeseries.format_written_timeseries_csv_message(sc, csv_path, n)
        ),
    )
    if result.skipped_positions:
        print(timeseries.format_skipped_positions_message(result.skipped_positions))
else:
    print("Skipped timeseries")

## Plot timeseries

In [ ]:
ts_dir = workspace_timeseries_dir(workspace)
plot_paths = plot_timeseries.run_plot_timeseries(
    metrics_dir=ts_dir,
    interval=INTERVAL_MINUTES,
    slide_channel_names=labels,
    columns=PLOT_COLUMNS,
)
for path in plot_paths:
    print(plot_timeseries.format_written_timeseries_plot_message(path))

## AUC

In [ ]:
auc_csv = auc.run_auc(workspace=workspace, interval=INTERVAL_MINUTES)
print(auc.format_written_auc_csv_message(auc_csv))

## Plot AUC

In [ ]:
auc_plots = plot_auc.run_plot_auc(auc_csv=auc_csv, slide_channel_names=labels)
for message in plot_auc.format_written_auc_plot_messages(list(auc_plots)):
    print(message)

## Fit

In [ ]:
fit_csv = fit.run_fit(
    workspace=workspace,
    interval=INTERVAL_MINUTES,
    max_onset_minutes=MAX_ONSET_MINUTES,
    jobs=JOBS,
)
print(fit.format_written_fit_csv_message(fit_csv))

## Plot fit

In [ ]:
fit_plots = plot_fit.run_plot_fit(
    fit_csv,
    slide_channel_names=labels,
    output=None,
    interval=INTERVAL_MINUTES,
    columns=PLOT_COLUMNS,
)
for message in plot_fit.format_written_fit_plot_messages(fit_plots):
    print(message)

print("Analysis finished.")